# Results — Model Comparison & Analysis

So sánh toàn bộ models, feature importance của LightGBM, và error analysis.

| Model | Loại |
|---|---|
| global_popularity | Baseline |
| recent_popularity | Baseline |
| age_segmented_popularity | Baseline |
| repurchase | Baseline |
| item_cf | Collaborative Filtering |
| two_stage_lgbm | Two-Stage LightGBM Ranker |


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
DATA = Path("../data")

articles     = pd.read_parquet(DATA / "articles.parquet")
customers    = pd.read_parquet(DATA / "customers.parquet")
transactions = pd.read_parquet(DATA / "transactions.parquet")


## 1. Chạy tất cả Models

In [ ]:
from ddm_project.baselines import (
    map_at_k,
    GlobalPopularityRecommender, RecentPopularityRecommender,
    RepurchaseRecommender, AgeSegmentedPopularityRecommender,
    train_test_split,
)
from ddm_project.cf_model import ItemCFRecommender, CFConfig
from ddm_project.model import TwoStageLGBMRanker, ModelConfig, make_splits

# --- splits ---
train_b, _, test_gt_b = train_test_split(transactions)   # for baselines & CF
train_full, train_feat, val_tx, _, test_gt = make_splits(transactions)

results: dict[str, float] = {}
SAMPLE = 50_000

# --- Baselines ---
for model in [
    GlobalPopularityRecommender(),
    RecentPopularityRecommender(),
    RepurchaseRecommender(),
    AgeSegmentedPopularityRecommender(),
]:
    if model.name == "age_segmented_popularity":
        model.fit(train_b, customers=customers)
    else:
        model.fit(train_b)
    results[model.name] = model.evaluate(test_gt_b, k=12, sample=SAMPLE)

# --- Item CF ---
cf = ItemCFRecommender(CFConfig(sample_eval=SAMPLE))
cf.fit(train_b)
results["item_cf"] = cf.evaluate(test_gt_b, k=12, sample=SAMPLE)

# --- Two-Stage LGBM ---
lgbm = TwoStageLGBMRanker(ModelConfig(sample_eval=SAMPLE))
lgbm.fit(train_feat, val_tx, customers, articles, train_full)
results["two_stage_lgbm"] = lgbm.evaluate(test_gt, k=12, sample=SAMPLE)

print("\nFinal scores:")
for name, s in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<40} MAP@12 = {s:.6f}")


## 2. Bảng So sánh MAP@12

In [ ]:
order = sorted(results.items(), key=lambda x: x[1])
names, scores = zip(*order)

colors = []
for n in names:
    if n == "two_stage_lgbm":
        colors.append("#2ecc71")
    elif n == "item_cf":
        colors.append("#3498db")
    else:
        colors.append("#95a5a6")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, scores, color=colors, edgecolor="white", linewidth=0.5)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.0003, bar.get_y() + bar.get_height() / 2,
            f"{score:.4f}", va="center", fontsize=10)

ax.set_xlabel("MAP@12", fontsize=12)
ax.set_title("Model Comparison — MAP@12 (higher is better)", fontsize=13)
ax.axvline(max(scores), color="green", linestyle="--", linewidth=1, alpha=0.5)

legend_patches = [
    mpatches.Patch(color="#2ecc71", label="Two-Stage LGBM (main model)"),
    mpatches.Patch(color="#3498db", label="Item CF"),
    mpatches.Patch(color="#95a5a6", label="Baselines"),
]
ax.legend(handles=legend_patches, loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison.png", bbox_inches="tight")
plt.show()

best_baseline = max(v for k, v in results.items() if k not in ["two_stage_lgbm", "item_cf"])
lgbm_score = results["two_stage_lgbm"]
print(f"\nTwo-Stage LGBM vs best baseline: +{(lgbm_score - best_baseline)/best_baseline:.1%}")


## 3. Feature Importance (LightGBM)

In [ ]:
from ddm_project.model import FEATURE_COLS

importance = lgbm.model.feature_importances_
feat_imp = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": importance,
}).sort_values("importance", ascending=False)

# Colour by feature group
def feat_group(name: str) -> str:
    if name.startswith("user_") or name in ["age", "age_group_enc", "FN", "Active", "club_active", "news_regular"]:
        return "User"
    if name.startswith("art_") or name in [
        "product_type_no", "product_group_enc", "graphical_appearance_no",
        "colour_group_code", "index_group_no", "section_no", "garment_group_no"
    ]:
        return "Article"
    return "User-Article"

feat_imp["group"] = feat_imp["feature"].apply(feat_group)
palette = {"User": "#4C72B0", "Article": "#DD8452", "User-Article": "#55A868"}
feat_imp["color"] = feat_imp["group"].map(palette)

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(feat_imp["feature"][::-1], feat_imp["importance"][::-1],
               color=feat_imp["color"][::-1])
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("LightGBM Feature Importance")

legend_patches = [mpatches.Patch(color=c, label=g) for g, c in palette.items()]
ax.legend(handles=legend_patches)
plt.tight_layout()
plt.savefig("feature_importance.png", bbox_inches="tight")
plt.show()

print("\nTop 10 features:")
print(feat_imp.head(10)[["feature", "group", "importance"]].to_string(index=False))


## 4. Error Analysis

In [ ]:
# Per-user AP@12 for LGBM model
from ddm_project.baselines import map_at_k

rng = np.random.default_rng(42)
sample_uids = rng.choice(list(test_gt.keys()), size=SAMPLE, replace=False).tolist()
gt_sub = {u: test_gt[u] for u in sample_uids}

preds = lgbm.predict(sample_uids, k=12)

def ap_at_k(predicted, actual, k=12):
    hits, score = 0, 0.0
    for i, p in enumerate(predicted[:k], 1):
        if p in actual:
            hits += 1
            score += hits / i
    return score / min(len(actual), k) if actual else 0.0

per_user_ap = pd.DataFrame({
    "customer_id": sample_uids,
    "ap12": [ap_at_k(preds.get(u, []), gt_sub[u]) for u in sample_uids],
})
per_user_ap = per_user_ap.merge(
    customers[["customer_id", "age"]].assign(
        age_group=pd.cut(customers["age"], bins=[15,25,35,45,55,65,100],
                         labels=["16-24","25-34","35-44","45-54","55-64","65+"])
    ),
    on="customer_id", how="left"
)

# Merge purchase count
tx_count = transactions[transactions["t_dat"] < (transactions["t_dat"].max() - pd.Timedelta(days=6))]
tx_count = tx_count.groupby("customer_id").size().rename("n_purchases").reset_index()
per_user_ap = per_user_ap.merge(tx_count, on="customer_id", how="left")
per_user_ap["purchase_bin"] = pd.cut(per_user_ap["n_purchases"].fillna(0),
                                      bins=[0,5,15,30,60,1000],
                                      labels=["1-5","6-15","16-30","31-60","61+"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# AP@12 by age group
ag = per_user_ap.dropna(subset=["age_group"])
ag.groupby("age_group", observed=True)["ap12"].mean().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="white"
)
axes[0].set_title("MAP@12 by Age Group")
axes[0].set_xlabel("Age Group")
axes[0].set_ylabel("MAP@12")
axes[0].tick_params(axis="x", rotation=0)

# AP@12 by purchase frequency
pb = per_user_ap.dropna(subset=["purchase_bin"])
pb.groupby("purchase_bin", observed=True)["ap12"].mean().plot(
    kind="bar", ax=axes[1], color="#DD8452", edgecolor="white"
)
axes[1].set_title("MAP@12 by Purchase History Size")
axes[1].set_xlabel("# Purchases in Training")
axes[1].set_ylabel("MAP@12")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("Error Analysis — Two-Stage LGBM", fontsize=13)
plt.tight_layout()
plt.savefig("error_analysis.png", bbox_inches="tight")
plt.show()

print("MAP@12 by age group:")
print(ag.groupby("age_group", observed=True)["ap12"].mean().round(4))
print("\nMAP@12 by purchase frequency:")
print(pb.groupby("purchase_bin", observed=True)["ap12"].mean().round(4))


In [ ]:
# Distribution of AP@12
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(per_user_ap["ap12"], bins=50, color="steelblue", edgecolor="white", linewidth=0.3)
zero_pct = (per_user_ap["ap12"] == 0).mean()
ax.set_xlabel("AP@12 per user")
ax.set_ylabel("# Users")
ax.set_title(f"Distribution of AP@12 — {zero_pct:.1%} users with score = 0 (cold-start)")
plt.tight_layout()
plt.savefig("ap_distribution.png", bbox_inches="tight")
plt.show()


## 5. Summary

| Model | Family | MAP@12 | ∆ vs. Repurchase | Ghi chú |
|:---|:---|---:|---:|:---|
| Global Popularity | Baseline | 0.0029 | -88.0% | Đơn giản nhất, hiệu quả thấp nhất |
| Age-segmented Popularity | Baseline | 0.0035 | -85.5% | Cải thiện nhẹ nhờ phân khúc nhân khẩu học |
| Recent Popularity (2w) | Baseline | 0.0068 | -71.8% | Hiệu quả hơn hẳn global nhờ nắm bắt trend ngắn hạn |
| Item-based CF | CF | 0.0086 | -64.3% | Tận dụng tín hiệu đồng mua sắm (co-occurrence) |
| Repurchase | Baseline | 0.0241 | - | Baseline cực mạnh — phản ánh thói quen mua sắm lặp lại |
| **Two-stage LightGBM** | **Main** | **0.0334** | **+38.6%** | **kết hợp đa dạng nguồn ứng viên & ranking** |


### **Key Insights:**

#### 1. Repurchase, Popularity and Trending Power:

- Repurchase là một baseline rất khó bị đánh bại (chiếm 30-40% giao dịch), cho thấy hành vi khách hàng có tính lặp lại cao.

- Tính thời thượng (Fashion Trend): Các đặc trưng ngắn hạn (art_pop_1w, art_trend_score) quan trọng hơn dài hạn (art_pop_4w). Intent của khách hàng thay đổi rất nhanh theo tuần.

#### 2. The Importance of Candidate Generation:

- Việc mở rộng ngân sách ứng viên từ 50 lên 200 đã giúp Recall@200 tăng vọt từ 9.6% lên 28%.

- Khả năng của bộ Ranker bị giới hạn bởi chất lượng của bộ lọc ứng viên (Retrieval). Nếu ứng viên tốt không được chọn từ đầu, model mạnh đến đâu cũng không thể gợi ý đúng.

#### 3. Key Feature Importance:

- Giá cả là chìa khóa: art_avg_price là feature quan trọng nhất. Kết hợp với các đặc trưng cá nhân hóa như ua_price_affinity giúp model học được "khẩu vị" túi tiền của từng khách hàng.

- Metadata đứng sau hành vi: Các thông tin tĩnh như màu sắc, section đứng thứ hạng thấp hơn so với các đặc trưng tương tác và thời gian.

#### 4. Cold-start challenges & Model success:

- 60% user có AP@12 = 0 (đa số là user mới, < 2 giao dịch). -> cold-start problem vẫn là một trong những thách thức lớn nhất

- Model hoạt động tốt nhất với nhóm 25-44 tuổi và có trên 15 giao dịch.

- Việc cải thiện 38.6% so với baseline Repurchase chứng minh model không chỉ "nhắc lại đồ cũ" mà đã thực sự gợi ý được những món đồ mới mà khách hàng yêu thích, tạo ra giá trị thặng dư (incremental value).